In [91]:
import pandas as pd
import seaborn as sns
from pathlib import Path
import numpy as np

In [3]:
from typing import Self

class BoundedFloat(float):
    def __new__(cls, x = 0) -> Self:
        if not ( 0 <= float(x) <= 1 ):
            raise ValueError( "Value of BoundedFloat must be between 0 and 1" +
                              f" received {x!r}" )
        return super().__new__(cls, x)
    
    def __init__(self, o: object) -> None:
        super().__init__()
        
PROJECT_ROOT = Path.cwd().parent
steam_games_dataset = PROJECT_ROOT / "datasets/steam-games-dataset/games.csv"

def make_subset(
    dataset_path: Path,
    subset_size: BoundedFloat = 0.2,
    reset_subset: bool = False,
) -> Path:
    outpath = dataset_path.with_stem(dataset_path.stem + "_subset")
    if outpath.exists() and not reset_subset:
        return outpath
    
    df = pd.read_csv(dataset_path)
    sub_size = int( len(df) * subset_size )
    sub_df = df[:sub_size]

    try:
        outpath.write_text(sub_df.to_csv())
        return outpath
    except Exception as e:
        print(f"{e!r}")
        raise

subset_path = make_subset(steam_games_dataset)
subset_df = pd.read_csv(subset_path)
print(subset_df.head().to_markdown())

|    |   Unnamed: 0 | AppID                                 | Name         | Release date   |   Estimated owners |   Peak CCU |   Required age |   Price |   DiscountDLC count | About the game                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                         

In [ ]:
# Data we're interested in: tags, genres, price, name, release date, metacritic score, user score, positive, negative, average/median playtime forever/two weeks
print("\n".join([c for c in subset_df.columns]))

Unnamed: 0
AppID
Name
Release date
Estimated owners
Peak CCU
Required age
Price
DiscountDLC count
About the game
Supported languages
Full audio languages
Reviews
Header image
Website
Support url
Support email
Windows
Mac
Linux
Metacritic score
Metacritic url
User score
Positive
Negative
Score rank
Achievements
Recommendations
Notes
Average playtime forever
Average playtime two weeks
Median playtime forever
Median playtime two weeks
Developers
Publishers
Categories
Genres
Tags
Screenshots
Movies


In [61]:
str(subset_df.iloc[2]["Genres"])

'Casual'

In [84]:
subset_df = subset_df.astype({
    "Price": float
})
(
    subset_df[
        subset_df["Genres"]
        .apply(
            lambda x: isinstance(x, str) and len(x.split(",")) < 3
        )
    ]
    .groupby(["Genres"])
    .agg({"Price": ["min", "max", "mean"]})
    .sort_values(("Price", "mean"), ascending=False)
    [:10]
)

Price                 
                                  min   max       mean
Genres                                                
Web Publishing                   75.0  75.0  75.000000
Sports,Education                 50.0  50.0  50.000000
Design & Illustration,Education   0.0  80.0  40.000000
Education,Software Training       0.0  70.0  38.000000
Action,Racing                     0.0  90.0  31.333333
Action,RPG                        0.0  95.0  28.652778
Action,Strategy                   0.0  90.0  27.830189
Action,Simulation                 0.0  92.0  25.000000
Simulation,Strategy               0.0  90.0  23.927928
Action,Adventure                  0.0  95.0  23.619186

In [123]:
full_df = pd.read_csv(steam_games_dataset)
full_df = full_df.astype({
    "Price": float
})
(
    full_df[
        full_df["Genres"]
        .apply(
            lambda x: isinstance(x, str) and len(x.split(",")) < 3
        )
    ]
    .groupby(["Genres"])
    .agg({"Price": ["mean","max","min","median"],
          "Metacritic score": ["mean"],
          "User score": ["mean"]})
    .sort_values(("Price", "mean"), ascending=False)
    [:10]
)

Price                     \
                                        mean   max   min median   
Genres                                                            
Massively Multiplayer,Sports       90.000000  90.0  90.0   90.0   
RPG,Indie                          72.500000  80.0  70.0   70.0   
Strategy,Simulation                68.000000  75.0  61.0   68.0   
Massively Multiplayer,Racing       55.500000  60.0  51.0   55.5   
Simulation,Design & Illustration   50.000000  50.0  50.0   50.0   
Audio Production,Game Development  45.000000  45.0  45.0   45.0   
Simulation,Strategy                36.968254  90.0   0.0   40.0   
Action,Utilities                   36.000000  80.0   0.0   25.0   
RPG,Sports                         35.000000  90.0   0.0   25.0   
Action,RPG                         34.344214  95.0   0.0   30.0   

                                  Metacritic score User score  
                                              mean       mean  
Genres                                                         
Massively Multiplayer,Sports               0.00000        0.0  
RPG,Indie                                 20.50000        0.0  
Strategy,Simulation                       37.50000        0.0  
Massively Multiplayer,Racing               0.00000        0.0  
Simulation,Design & Illustration           0.00000        0.0  
Audio Production,Game Development          0.00000        0.0  
Simulation,Strategy                       13.03351        0.0  
Action,Utilities                           0.00000        0.0  
RPG,Sports                                 0.00000        0.0  
Action,RPG                                18.84273        0.0

In [127]:
full_df.iloc[0]["Negative"]

np.int64(0)

In [130]:
from collections import Counter
c = Counter([g for g in full_df["Genres"] if isinstance(g, str)])
f = [k for k, v in c.items() if v >= 500]
(
    full_df[
        full_df["Genres"]
        .apply(
            lambda x: isinstance(x, str) and x in f
        )
        & (full_df["Price"] > 0)
    ]
    .groupby(["Genres"])
    .agg({"Price": ["mean","max","min","median"],
          "Metacritic score": ["mean", "max", "min"],
        #   "User score": ["mean", "max", "min"]})
    })
    .sort_values(("Price", "mean"), ascending=False)
    # [:10]
)

Price                      \
                                          mean    max   min median   
Genres                                                               
Action,Adventure                     62.601423   95.0  10.0   67.0   
Simulation,Strategy                  61.469208   90.0  10.0   67.0   
Action,Indie,Strategy                60.878788   93.0  10.0   60.0   
Action                               60.189588  100.0  10.0   60.0   
Action,Adventure,RPG                 59.887273   95.0  10.0   60.0   
Strategy                             59.300813   90.0  10.0   60.0   
Action,Adventure,Indie               59.287603   95.0  10.0   60.0   
Action,Indie                         59.125353  100.0  10.0   60.0   
Action,Adventure,Casual,Indie        58.625162   95.0  10.0   60.0   
Action,Adventure,Indie,RPG           58.577424   95.0  10.0   60.0   
Indie,Simulation,Strategy            57.803318   95.0  10.0   60.0   
Adventure,Indie                      57.653846   95.0  10.0   55.0   
Action,Casual                        57.240196   95.0  20.0   60.0   
Indie,Strategy                       57.116228  100.0  10.0   59.0   
Action,Casual,Indie                  57.029235   95.0  10.0   51.0   
Indie,RPG,Strategy                   56.696203   95.0  10.0   55.0   
Indie,Simulation                     56.654639  100.0  10.0   51.0   
Adventure,Casual,Indie               56.409429   95.0  10.0   51.0   
Action,Indie,RPG                     56.225914   92.0  10.0   55.0   
Action,Adventure,Casual,Indie,RPG    55.884211   95.0  15.0   51.0   
Adventure,Casual                     55.871003   95.0  10.0   50.0   
Adventure                            55.534091   95.0  10.0   50.0   
Indie                                55.292159   92.0  10.0   51.0   
Adventure,Casual,Indie,Simulation    54.793388   95.0  10.0   51.0   
RPG                                  54.763538   93.0  10.0   50.0   
Action,Casual,Indie,Strategy         54.747126   95.0  10.0   51.0   
Adventure,Casual,Indie,RPG           54.298295   95.0  10.0   50.0   
Casual,Indie,Free To Play            54.000000   60.0  51.0   51.0   
Casual,Indie,Strategy                53.458182   95.0  10.0   50.0   
Simulation                           53.392578   95.0  10.0   50.0   
Casual,Indie                         53.282169   95.0  10.0   51.0   
Casual                               52.273302   95.0  10.0   50.0   
Adventure,Indie,Simulation           52.089172   92.0  10.0   50.0   
Casual,Indie,Simulation,Strategy     51.773196   95.0  10.0   50.0   
Adventure,RPG                        51.305164   91.0  10.0   50.0   
Indie,RPG                            51.168831   94.0  10.0   50.0   
Casual,Strategy                      50.299435   90.0  15.0   50.0   
Adventure,Indie,RPG                  49.870396  100.0  10.0   50.0   
Casual,Indie,Simulation              49.496192   95.0  10.0   50.0   
Casual,Simulation                    49.257042   90.0  10.0   50.0   
Action,Adventure,Indie,Early Access  42.834586   90.0  10.0   40.0   
Action,Indie,Early Access            42.152318   90.0  10.0   35.0   

                                    Metacritic score          
                                                mean max min  
Genres                                                        
Action,Adventure                           19.049822  95   0  
Simulation,Strategy                        18.806452  91   0  
Action,Indie,Strategy                      10.194805  89   0  
Action                                     15.303030  96   0  
Action,Adventure,RPG                       15.072727  93   0  
Strategy                                   18.744715  94   0  
Action,Adventure,Indie                      7.644029  90   0  
Action,Indie                                6.586711  91   0  
Action,Adventure,Casual,Indie               2.902724  90   0  
Action,Adventure,Indie,RPG                  9.318379  88   0  
Indie,Simulation,Strategy                   9.232227  93   0  
Adventur